# Análisis de Housing con Spark SQL

Este notebook realiza el ejercicio utilizando **PySpark** y consultas mediante `spark.sql(...)`.

Actividades:

1. Configuración de la plataforma Spark.
2. Importación del archivo de Housing a un DataFrame de Spark.
3. Registro del DataFrame como tabla temporal.
4. Listado completo de columnas ordenado por `zipcode`.
5. Identificación del `zipcode` con mayor número de casas.
6. Cálculo del precio promedio y tamaño habitable promedio en m² para ese código postal.
7. Agrupamiento por `zipcode`, habitaciones y baños, calculando el precio promedio.


In [ ]:
# 1) Configurar Java y el entorno de Spark antes de importar PySpark

import os
import sys
from pathlib import Path

CONDA_PREFIX = Path(sys.prefix)

# Java instalado dentro del entorno de Anaconda
java_home = CONDA_PREFIX / "Library"

if not (java_home / "bin" / "java.exe").exists():
    raise FileNotFoundError(
        "No se encontró Java dentro del entorno de Anaconda.\n"
        "Abre Anaconda Prompt, activa el entorno spark y ejecuta:\n"
        "conda install -c conda-forge openjdk=8"
    )

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

Python: c:\Users\RyanHz\anaconda3\envs\spark\python.exe
JAVA_HOME: c:\Users\RyanHz\anaconda3\envs\spark\Library


In [ ]:
# 2) Verificar la versión de PySpark

import pyspark

print("Versión de PySpark:", pyspark.__version__)
print("Ubicación de PySpark:", pyspark.__file__)

Versión de PySpark: 3.5.6
Ubicación de PySpark: c:\Users\RyanHz\anaconda3\envs\spark\Lib\site-packages\pyspark\__init__.py


In [ ]:
# 3) Crear la sesión de Spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HousingSparkSQL")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

print("Spark iniciado correctamente")
print("Versión de Spark:", spark.version)

Spark iniciado correctamente
Versión de Spark: 3.5.6


## Importación de los datos de Housing

In [ ]:
# 4) Localizar el archivo CSV

from pathlib import Path

posibles_archivos = [
    Path("../../../Archivos-Analisis/files-tarea-m40/kc_house_data.csv")
]

ruta_csv = next((ruta for ruta in posibles_archivos if ruta.exists()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró el CSV. Coloca 'kc_house_data(1).csv' "
        "en la misma carpeta que este notebook."
    )

print("Archivo encontrado:", ruta_csv.resolve())

Archivo encontrado: C:\Users\RyanHz\Documents\EBAC\VS\Archivos-Analisis\files-tarea-m40\kc_house_data.csv


In [ ]:
# 5) Cargar el CSV en un DataFrame de Spark

housing_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

print("Número de registros:", housing_df.count())
print("Número de columnas:", len(housing_df.columns))

housing_df.show(5, truncate=False)

Número de registros: 21613
Número de columnas: 21
+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|id        |date           |price   |bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|lat    |long    |sqft_living15|sqft_lot15|
+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|7129300520|20141013T000000|221900.0|3       |1.0      |1180       |5650    |1.0   |0         |0   |3        |7    |1180      |0            |1955    |0           |98178  |47.5112|-122.257|1340         |5650      |
|6414100192|20141209T000000|538000.0|3       |2.25     |2570       |7242    |2.0   |0         

In [ ]:
# Revisar tipos de datos y estructura

housing_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft_living: integer (nullable = true)
 |-- sqft_lot: integer (nullable = true)
 |-- floors: double (nullable = true)
 |-- waterfront: integer (nullable = true)
 |-- view: integer (nullable = true)
 |-- condition: integer (nullable = true)
 |-- grade: integer (nullable = true)
 |-- sqft_above: integer (nullable = true)
 |-- sqft_basement: integer (nullable = true)
 |-- yr_built: integer (nullable = true)
 |-- yr_renovated: integer (nullable = true)
 |-- zipcode: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- sqft_living15: integer (nullable = true)
 |-- sqft_lot15: integer (nullable = true)



In [ ]:
# 6) Crear una tabla temporal para utilizar Spark SQL

housing_df.createOrReplaceTempView("housing")

print("Tabla temporal 'housing' creada correctamente.")

Tabla temporal 'housing' creada correctamente.


## Consulta 1: listado completo ordenado por `zipcode`

La consulta selecciona todas las columnas del conjunto de datos y ordena los registros por código postal.

In [ ]:
consulta_1 = """
SELECT *
FROM housing
ORDER BY zipcode ASC
"""

housing_ordenado = spark.sql(consulta_1)

housing_ordenado.show(50, truncate=False)

+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|id        |date           |price   |bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|lat    |long    |sqft_living15|sqft_lot15|
+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|303000445 |20140523T000000|175000.0|2       |1.0      |1300       |44431   |1.0   |0         |0   |5        |6    |1300      |0            |1958    |0           |98001  |47.327 |-122.267|1470         |14850     |
|6085000130|20150321T000000|230000.0|3       |1.0      |1140       |9639    |1.0   |0         |0   |4        |7    |1140      |0            |196

## Consulta 2: `zipcode` con mayor número de casas

Primero se cuenta el número de viviendas por código postal. Después se ordena el resultado de mayor a menor.

In [ ]:
consulta_conteo_zipcode = """
SELECT
    zipcode,
    COUNT(*) AS numero_casas
FROM housing
GROUP BY zipcode
ORDER BY numero_casas DESC, zipcode ASC
"""

conteo_por_zipcode = spark.sql(consulta_conteo_zipcode)

conteo_por_zipcode.show(20, truncate=False)

+-------+------------+
|zipcode|numero_casas|
+-------+------------+
|98103  |602         |
|98038  |590         |
|98115  |583         |
|98052  |574         |
|98117  |553         |
|98042  |548         |
|98034  |545         |
|98118  |508         |
|98023  |499         |
|98006  |498         |
|98133  |494         |
|98059  |468         |
|98058  |455         |
|98155  |446         |
|98074  |441         |
|98033  |432         |
|98027  |412         |
|98125  |410         |
|98056  |406         |
|98053  |405         |
+-------+------------+
only showing top 20 rows



In [ ]:
# Obtener el zipcode con mayor número de casas

consulta_zipcode_mayor = """
SELECT
    zipcode,
    COUNT(*) AS numero_casas
FROM housing
GROUP BY zipcode
ORDER BY numero_casas DESC, zipcode ASC
LIMIT 1
"""

zipcode_mayor_df = spark.sql(consulta_zipcode_mayor)
zipcode_mayor_df.show()

zipcode_mayor = zipcode_mayor_df.first()["zipcode"]

print("Zipcode con mayor número de casas:", zipcode_mayor)

+-------+------------+
|zipcode|numero_casas|
+-------+------------+
|  98103|         602|
+-------+------------+

Zipcode con mayor número de casas: 98103


## Precio promedio y tamaño promedio en m²

El campo `sqft_living` está expresado en pies cuadrados.

La conversión utilizada es:

\[
1\text{ pie}^2 = 0.092903\text{ m}^2
\]


In [ ]:
consulta_promedios = f"""
SELECT
    zipcode,
    COUNT(*) AS numero_casas,
    ROUND(AVG(price), 2) AS precio_promedio,
    ROUND(AVG(sqft_living * 0.092903), 2) AS tamanio_promedio_m2
FROM housing
WHERE zipcode = {zipcode_mayor}
GROUP BY zipcode
"""

promedios_zipcode_mayor = spark.sql(consulta_promedios)

promedios_zipcode_mayor.show(truncate=False)

+-------+------------+---------------+-------------------+
|zipcode|numero_casas|precio_promedio|tamanio_promedio_m2|
+-------+------------+---------------+-------------------+
|98103  |602         |584919.21      |153.37             |
+-------+------------+---------------+-------------------+



## Consulta 3: agrupamiento por habitaciones, baños y `zipcode`

El resultado muestra:

- Código postal.
- Número de habitaciones.
- Número de baños.
- Precio promedio.
- Número de viviendas que forman cada grupo.


In [ ]:
consulta_agrupamiento = """
SELECT
    zipcode,
    bedrooms,
    bathrooms,
    ROUND(AVG(price), 2) AS precio_promedio,
    COUNT(*) AS numero_casas
FROM housing
GROUP BY
    zipcode,
    bedrooms,
    bathrooms
ORDER BY
    zipcode ASC,
    bedrooms ASC,
    bathrooms ASC
"""

agrupamiento_housing = spark.sql(consulta_agrupamiento)

agrupamiento_housing.show(100, truncate=False)

+-------+--------+---------+---------------+------------+
|zipcode|bedrooms|bathrooms|precio_promedio|numero_casas|
+-------+--------+---------+---------------+------------+
|98001  |0       |0.0      |139950.0       |1           |
|98001  |1       |1.0      |166000.0       |1           |
|98001  |1       |2.0      |171000.0       |2           |
|98001  |2       |1.0      |197428.57      |14          |
|98001  |2       |1.5      |350000.0       |1           |
|98001  |2       |1.75     |246112.5       |4           |
|98001  |2       |2.5      |214100.0       |1           |
|98001  |2       |2.75     |239475.0       |2           |
|98001  |3       |0.75     |363000.0       |1           |
|98001  |3       |1.0      |205182.81      |42          |
|98001  |3       |1.5      |224108.5       |20          |
|98001  |3       |1.75     |260531.08      |37          |
|98001  |3       |2.0      |256841.43      |28          |
|98001  |3       |2.25     |265999.0       |12          |
|98001  |3    

## Consultas de comprobación

In [ ]:
# Comprobar que la tabla temporal contiene todos los registros

spark.sql("""
SELECT COUNT(*) AS total_registros
FROM housing
""").show()

+---------------+
|total_registros|
+---------------+
|          21613|
+---------------+



In [ ]:
# Mostrar las columnas utilizadas en el ejercicio

spark.sql("""
SELECT
    zipcode,
    price,
    sqft_living,
    bedrooms,
    bathrooms
FROM housing
ORDER BY zipcode
LIMIT 20
""").show(truncate=False)

+-------+--------+-----------+--------+---------+
|zipcode|price   |sqft_living|bedrooms|bathrooms|
+-------+--------+-----------+--------+---------+
|98001  |227950.0|1670       |3       |1.5      |
|98001  |267000.0|2495       |3       |2.5      |
|98001  |196000.0|2070       |3       |2.25     |
|98001  |233000.0|1400       |3       |2.0      |
|98001  |320000.0|2280       |3       |2.5      |
|98001  |249900.0|1380       |3       |1.75     |
|98001  |287000.0|2240       |4       |2.5      |
|98001  |205000.0|1410       |3       |2.0      |
|98001  |252000.0|1550       |4       |1.5      |
|98001  |196500.0|1320       |3       |1.0      |
|98001  |360000.0|2160       |4       |2.5      |
|98001  |199950.0|1590       |2       |2.75     |
|98001  |305000.0|2250       |4       |2.5      |
|98001  |289000.0|1850       |3       |2.0      |
|98001  |240000.0|1220       |4       |1.0      |
|98001  |249900.0|2080       |3       |1.75     |
|98001  |465000.0|2714       |3       |2.5      |


## Cerrar la sesión de Spark

Ejecuta la siguiente celda únicamente cuando hayas terminado de trabajar con el notebook.

In [ ]:
spark.stop()
print("Sesión de Spark cerrada correctamente.")

Sesión de Spark cerrada correctamente.
